# Investment: Build-Timing with Limited Lifetime

`Investment` models a **singular build decision**: the solver decides
**when** to build and at **what size**. Once built, capacity is available
for a limited number of periods (the **lifetime**), then retires.

This is different from `Sizing`, where each period independently chooses
its capacity. Investment answers: *"In which period should I build this
plant, given that it only lasts N periods?"*

This notebook demonstrates:

1. **Lifetime retirement** — a cheap plant that expires, forcing fallback to an expensive alternative
2. **Build-timing optimization** — the solver picks the optimal period to build
3. **Prior capacity** — pre-existing plants available from period 0

In [ ]:
from datetime import datetime

import numpy as np
import pandas as pd

from fluxopt import Carrier, Effect, Flow, Investment, Port, optimize

## Setup

A factory needs 10 MW of heat across four 5-year planning periods.
Two supply options:

| Source | Cost | Investment | Lifetime |
|---|---|---|---|
| **Cheap plant** | 1 EUR/MWh | build once, 0 CAPEX | 2 periods (10 years) |
| **Expensive backup** | 50 EUR/MWh | always available | -- |

The cheap plant can only be built once and lasts 2 periods. After it
retires, the factory must fall back on the expensive backup.

In [ ]:
timesteps = [datetime(2025, 1, 15, h) for h in range(6, 10)]
periods = [2025, 2030, 2035, 2040]
period_weights = [5, 5, 5, 5]  # each period represents 5 years
demand = [10, 10, 10, 10]  # constant 10 MW

## Lifetime = 2 periods

The cheap plant has `lifetime=2`: once built, it is active for 2 periods,
then retires. The solver must decide *when* to build it.

In [ ]:
result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('heat')],
    effects=[Effect('cost', unit='EUR', is_objective=True)],
    ports=[
        Port(
            'factory',
            exports=[Flow('heat', size=1, fixed_relative_profile=demand)],
        ),
        Port(
            'cheap_plant',
            imports=[
                Flow(
                    'heat',
                    short_id='cheap',
                    size=Investment(0, 20, lifetime=2),
                    effects_per_flow_hour={'cost': 1},
                ),
            ],
        ),
        Port(
            'backup',
            imports=[
                Flow('heat', short_id='expensive', effects_per_flow_hour={'cost': 50}),
            ],
        ),
    ],
    periods=periods,
    period_weights=period_weights,
)

### When does the solver build?

The `invest--build` variable shows the build period. The `invest--active`
variable shows which periods the plant is operational:

In [ ]:
sol = result.solution
invest_flow = 'cheap_plant(cheap)'

build = sol['invest--build'].sel(flow=invest_flow).values
active = sol['invest--active'].sel(flow=invest_flow).values
size = sol['invest--size'].sel(flow=invest_flow).values

pd.DataFrame(
    {'build': build.astype(int), 'active': active.astype(int)},
    index=pd.Index(periods, name='period'),
).assign(invest_size=float(size))

### Dispatch per period

The cheap plant handles demand while active. After retirement, the
expensive backup takes over:

In [ ]:
rates = sol['flow--rate']

# Average flow rate per period for each source
cheap_rate = rates.sel(flow=invest_flow).mean('time')
expensive_rate = rates.sel(flow='backup(expensive)').mean('time')

pd.DataFrame(
    {
        'cheap (MW)': cheap_rate.values.round(1),
        'expensive (MW)': expensive_rate.values.round(1),
    },
    index=pd.Index(periods, name='period'),
)

### Cost breakdown

Costs are low while the cheap plant is active, then jump when the backup
takes over:

In [ ]:
totals = result.effect_totals.sel(effect='cost')

pd.DataFrame(
    {
        'cost/period (EUR)': totals.values.round(0),
        'weight': period_weights,
        'weighted (EUR)': (totals.values * np.array(period_weights)).round(0),
    },
    index=pd.Index(periods, name='period'),
).assign(objective=result.objective)

## Comparing lifetimes

What happens when the plant lasts longer? Let's sweep lifetime from 1 to 4
periods and see how the total cost changes:

In [ ]:
rows = []
for lt in [1, 2, 3, 4]:
    r = optimize(
        timesteps=timesteps,
        carriers=[Carrier('heat')],
        effects=[Effect('cost', is_objective=True)],
        ports=[
            Port('factory', exports=[Flow('heat', size=1, fixed_relative_profile=demand)]),
            Port(
                'cheap_plant',
                imports=[
                    Flow(
                        'heat',
                        short_id='cheap',
                        size=Investment(0, 20, lifetime=lt),
                        effects_per_flow_hour={'cost': 1},
                    ),
                ],
            ),
            Port(
                'backup',
                imports=[Flow('heat', short_id='expensive', effects_per_flow_hour={'cost': 50})],
            ),
        ],
        periods=periods,
        period_weights=period_weights,
    )
    s = r.solution
    active_periods = int(s['invest--active'].sel(flow=invest_flow).sum().values)
    build_period = periods[int(s['invest--build'].sel(flow=invest_flow).values.argmax())]
    rows.append(
        {
            'lifetime': f'{lt} period(s)',
            'build_in': build_period,
            'active_periods': active_periods,
            'objective': round(r.objective, 0),
        }
    )

pd.DataFrame(rows).set_index('lifetime')

Longer lifetime means more periods covered by the cheap plant,
fewer periods on the expensive backup, and a lower total cost.
With `lifetime=4` the cheap plant covers all periods.

## Prior capacity

`prior_size` models a plant that already exists at the start of the
planning horizon. No build decision needed — capacity is available from
period 0, subject to the same lifetime rules.

In [ ]:
result_prior = optimize(
    timesteps=timesteps,
    carriers=[Carrier('heat')],
    effects=[Effect('cost', is_objective=True)],
    ports=[
        Port('factory', exports=[Flow('heat', size=1, fixed_relative_profile=demand)]),
        Port(
            'existing_plant',
            imports=[
                Flow(
                    'heat',
                    short_id='existing',
                    size=Investment(0, 20, mandatory=False, prior_size=10, lifetime=2),
                    effects_per_flow_hour={'cost': 1},
                ),
            ],
        ),
        Port(
            'backup',
            imports=[Flow('heat', short_id='expensive', effects_per_flow_hour={'cost': 50})],
        ),
    ],
    periods=periods,
    period_weights=period_weights,
)

sol_prior = result_prior.solution
prior_flow = 'existing_plant(existing)'

active_prior = sol_prior['invest--active'].sel(flow=prior_flow).values
build_prior = sol_prior['invest--build'].sel(flow=prior_flow).values
flow_size_prior = sol_prior['flow--size'].sel(flow=prior_flow).values

pd.DataFrame(
    {
        'build': build_prior.astype(int),
        'active': active_prior.astype(int),
        'flow_size (MW)': flow_size_prior.round(1),
    },
    index=pd.Index(periods, name='period'),
)

The existing plant is active for the first 2 periods (its lifetime),
then retires. No new build happens (`mandatory=False` and it's cheaper
to let the backup handle the remaining periods than to rebuild).

## How it works

Investment adds three decision variables per flow:

| Variable | Dims | Meaning |
|---|---|---|
| `invest_size` | `(flow,)` | Capacity chosen once (period-independent) |
| `invest_build` | `(flow, period)` | Binary: build in this period? |
| `invest_active` | `(flow, period)` | Binary: operational in this period? |

The existing `flow_size` variable is linked to `invest_size * active`
via big-M constraints, so it equals the invested capacity when active
and zero when retired. Flow rate bounds (`P <= flow_size * rel_ub`)
apply automatically.

**Key constraints:**
- **Build-once**: `sum_p build[p] == 1` (mandatory) or `<= 1` (optional)
- **Active logic**: `active[p] = sum_{tau: p in [tau, tau+lifetime)} build[tau]`
- **Prior capacity**: `active[p] = 1` for `p < lifetime` when `prior_size > 0`